# 🚀 Notebook do Professor (Demo) — Aula 01: Revisão expressa + LangChain LCEL e ChatOllama

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 01/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🐍 LCEL · ChatOllama · OutputParser**  
**🔁 Andaime 40%**  

---

## 🎯 Objetivo da aula

Construir uma chain LangChain com LCEL que substitui o chamar_llm() manual do 1º semestre — com menos código, mais composição e pronto para escalar com memória e RAG nas próximas aulas.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — O ponto de partida: o chatbot manual do 1º semestre

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
def montar_prompt(historico, pergunta):
    return [{"role": "system", "content": SYSTEM_PROMPT}] \
        + historico + [{"role": "user", "content": pergunta}]

def chamar_llm(messages):
    resp = ollama.chat(model="gpt-oss:120b", messages=messages)
    return resp["message"]["content"]

def iniciar_chat():
    historico = []
    while True:
        pergunta = input("Você: ")
        if pergunta == "sair": break
        msgs = montar_prompt(historico, pergunta)
        resposta = chamar_llm(msgs)
        historico += [{"role": "user", "content": pergunta},
                     {"role": "assistant", "content": resposta}]
        print("Bot:", resposta)

### Slide 11 — LCEL — o operador | como pipe

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Cada componente é independente — compostos pelo operador |

# 1. Template: define estrutura e variáveis do prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é {persona}. Responda sobre {especialidade}."),
    ("human",  "{pergunta}"),
])

# 2. Modelo: conecta ao Ollama Cloud
llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
)

# 3. Parser: extrai só o texto da resposta
parser = StrOutputParser()

# Composição com |  — isso é uma Runnable, não uma chamada
chain = prompt | llm | parser

# Invocar a chain com as variáveis do template
resposta = chain.invoke({
    "persona":       "um chef de culinária brasileira",
    "especialidade": "culinária brasileira",
    "pergunta":      "Como faço um bolo de cenoura?",
})
print(resposta)  # → string direta, sem resp["message"]["content"]

### Slide 12 — ChatPromptTemplate — evolução do montar_prompt()

In [ ]:
def montar_prompt(pergunta,
                    especialidade,
                    tom="direto"):
    return f"""
Você é especialista em {especialidade}.
Pergunta: {pergunta}
Tom: {tom}.
"""

# Problema: f-string não valida variáveis
# se esquecer uma, silencia sem erro

### Slide 12 — ChatPromptTemplate — evolução do montar_prompt()

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Você é especialista em {especialidade}."
     " Tom: {tom}."),
    ("human", "Pergunta: {pergunta}"),
])

# Vantagem: valida variáveis automaticamente
# KeyError se esquecer alguma ao invocar
# Separação de roles limpa (system/human/ai)

### Slide 13 — Output Parsers — StrOutputParser e JsonOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

# Retorna string direta
resposta = chain.invoke({"pergunta":"Oi"})
print(type(resposta))  # → <class 'str'>

# Sem mais resp["message"]["content"] !

### Slide 13 — Output Parsers — StrOutputParser e JsonOutputParser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | llm | JsonOutputParser()

# O prompt DEVE pedir JSON explicitamente
resultado = chain.invoke({
    "pergunta": "Liste 3 ingredientes"
})
print(type(resultado))   # → <class 'dict'>
print(resultado["ingredientes"])

### Slide 14 — ChatOllama — conectando ao Ollama Cloud

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Instanciar o modelo — parâmetros declarativos
llm = ChatOllama(
    model="gpt-oss:120b",          # modelo padrão do semestre
    temperature=0.7,               # mesmo parâmetro do 1º sem
    num_predict=512,               # equivalente a max_tokens
)

# Teste rápido direto (sem chain)
resp = llm.invoke("Olá! Responda em 1 linha.")
print(resp.content)   # .content — não ["message"]["content"]
print(type(resp))      # → AIMessage (objeto LangChain)

### Slide 24 — Resumo — o que é novo no Python desta aula

In [ ]:
# 1. Instância declarativa com keyword arguments
llm = ChatOllama(model="gpt-oss:120b", temperature=0.7)

# 2. Método de classe com lista de tuplas
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é {persona}."),
    ("human",  "{pergunta}"),
])

# 3. Operator overloading — | cria chain composta
chain = prompt | llm | StrOutputParser()

# 4. .invoke() — método com dict de variáveis
resp = chain.invoke({"persona": "chef", "pergunta": "Olá"})

# 5. Generator com for + yield (em .stream())
for chunk in chain.stream({"pergunta": "Oi"}):
    print(chunk, end="", flush=True)
    # flush=True força exibição imediata de cada token

# 6. os.environ — variáveis de ambiente em Python
import os
os.environ["OLLAMA_API_KEY"] = "valor"  # dict especial do SO

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 01 · 2º Semestre**  
### Construir a chain base do grupo ★★

*Grupo 3–4 · 20 minutos · Google Colab*

1. Escolha o domínio do grupo — este domínio vai persistir pelo semestre inteiro: CKP01 (chatbot LangChain), CKP02 (RAG sobre documentos desse domínio), CKP03 (agente). Exemplos: culinária, direito do consumidor, saúde, games, finanças pessoais.
2. Complete as 4 lacunas — modelo, template com persona do domínio, composição com | e chamada com .invoke().
3. Teste com 3 perguntas do domínio — confirme que a persona está coerente com o domínio escolhido.
4. Desafio — JsonOutputParser: crie uma segunda chain com JsonOutputParser. No prompt, peça ao modelo para responder em JSON com as chaves resposta e confianca (0–10). Imprima o tipo do resultado.

> **🎯 Orientação das lacunas**
>
> Lacuna 1:  nome do modelo do semestre e a temperatura usada desde o 1º semestre (0.7).
>
> Lacuna 2:  a role de sistema recebe a persona e o domínio do grupo; a role humana recebe a variável da pergunta.
>
> Lacuna 3:  os 3 componentes na ordem prompt → modelo → parser, unidos pelo operador pipe.
>
> Lacuna 4:  qualquer pergunta real do domínio escolhido pelo grupo.

> **💡 Dica para o desafio:**
>
> no prompt, adicione: "Responda SOMENTE em JSON no formato: {\"resposta\": \"...\", \"confianca\": N}". O modelo precisa da instrução explícita para gerar JSON válido.

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
import os

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: instancie o ChatOllama com o modelo certo
llm = ChatOllama(model=___, temperature=___)

# 👉 LACUNA 2: crie o ChatPromptTemplate com 2 roles
# system: persona + especialidade do grupo
# human: a pergunta do usuário
prompt = ChatPromptTemplate.from_messages([
    (___, ___),  # system com persona do grupo
    (___, ___),  # human com {pergunta}
])

# 👉 LACUNA 3: componha a chain com os 3 componentes
chain = ___ | ___ | ___

# 👉 LACUNA 4: invoque com uma pergunta do domínio do grupo
resposta = chain.invoke({"pergunta": ___})
print(resposta)

# Bônus: testar .stream() — observe os tokens chegando
for chunk in chain.stream({"pergunta": "Me dê um exemplo rápido."}):
    print(chunk, end="", flush=True)

## 📚 Referências da aula

- Docs LangChain — LCEL (LangChain Expression Language): composição declarativa de chains com o operador |. python.langchain.com/docs/concepts/lcel
- Docs LangChain — ChatPromptTemplate: estruturar prompts com roles e variáveis. python.langchain.com/docs/concepts/prompt_templates
- Docs LangChain — Output parsers: StrOutputParser, JsonOutputParser, PydanticOutputParser. python.langchain.com/docs/concepts/output_parsers
- Docs langchain-ollama — ChatOllama: integração LangChain com modelos Ollama. python.langchain.com/docs/integrations/chat/ollama
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2: Agentes racionais — o modelo de percepção → ação que fundamenta o conceito de pipeline de LLM.
- Ebook Polzer, D. — RAG with Python Cookbook. O'Reilly, 2025. Cap. 1: escolha de frameworks para aplicações RAG e por que LangChain (LCEL) é o padrão de mercado para orquestração.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 1: Prompt Chaining — a base dos padrões agênticos; o LCEL que você aprendeu hoje é a fundação dos agentes da Aula 10.
- Ebook Lanham, M. — AI Agents in Action. Manning, 2025. Cap. 2: prompting LLMs com personas e delimitadores — reforça o que você construiu no 1º semestre.

---

**→ Próxima Aula — Aula 02 · 10/08** — Memória conversacional — Buffer, Summary e TokenBuffer
  
A chain de hoje é stateless — cada .invoke() começa do zero, igual à lista historico manual do 1º semestre, que crescia sem limite de tokens. A Aula 02 resolve isso com 3 tipos de memória — Buffer, Summary e TokenBuffer — e o CKP01 fica mais próximo.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*